# Predicting heart disease using machine learning

This notebook looks into using various Python-based machine learning and data science libraries in an attempt to build a machine learning model capable of predicting whether or not someone has heart disease based on their medical attributes.

We're going to take the following approach:
1. Problem definition
2. Data
3. Evaluation
4. Features
5. Modelling
6. Experimentation

## 1. Problem Definition

In a statement,
> Given clinical parameters about a patient, can we predict whether or not they have heart disease?

## 2. Data

The original data came from the Cleveland data from the UCI Machine Learning Repository. https://archive.ics.uci.edu/ml/datasets/heart+Disease

There is also a version of it available on Kaggle. https://www.kaggle.com/datasets/sumaiyatasmeem/heart-disease-classification-dataset

## 3. Evaluation

> If we can reach 95% accuracy at predicting whether or not a patient has heart disease during the proof of concept, we'll pursue the project.

## 4. Features

This is where you'll get different information about each of the features in your data. You can do this via doing your own research (such as looking at the links above) or by talking to a subject matter expert (someone who knows about the dataset).

**Create data dictionary**

1. age - age in years
2. sex - (1 = male; 0 = female)
3. cp - chest pain type
    * 0: Typical angina: chest pain related decrease blood supply to the heart
    * 1: Atypical angina: chest pain not related to heart
    * 2: Non-anginal pain: typically esophageal spasms (non heart related)
    * 3: Asymptomatic: chest pain not showing signs of disease
4. trestbps - resting blood pressure (in mm Hg on admission to the hospital) anything above 130-140 is typically cause for concern
5. chol - serum cholesterol in mg/dl
    * serum = LDL + HDL + .2 * triglycerides
    * above 200 is cause for concern
6. fbs - (fasting blood sugar > 120 mg/dl) (1 = true; 0 = false)
    * '>126' mg/dL signals diabetes
7. restecg - resting electrocardiographic results
    * 0: Nothing to note
    * 1: ST-T Wave abnormality
        * can range from mild symptoms to severe problems
        * signals non-normal heart beat
    * 2: Possible or definite left ventricular hypertrophy
        * Enlarged heart's main pumping chamber
8. thalach - maximum heart rate achieved
9. exang - exercise induced angina (1 = yes; 0 = no)
10. oldpeak - ST depression induced by exercise relative to rest looks at stress of heart during exercise; unhealthy heart will stress more
11. slope - the slope of the peak exercise ST segment
    * 0: Upsloping: better heart rate with exercise (uncommon)
    * 1: Flatsloping: minimal change (typical healthy heart)
    * 2: Downsloping: signs of unhealthy heart
12. ca - number of major vessels (0-3) colored by fluoroscopy
    * colored vessel means the doctor can see the blood passing through
    * the more blood movement the better (no clots)
13. thal - thallium stress result
    * 1,3: normal
    * 6: fixed defect: used to be defect but ok now
    * 7: reversible defect: no proper blood movement when exercising
14. target - have disease or not (1=yes, 0=no) (= the predicted attribute)


In [2]:
# Imports for the project
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

# Make plots visible inline
%matplotlib inline
#Seaborn Customisation
sns.set_theme(style="darkgrid")

# Importing models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Model Evaluations
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pl.read_csv("data/heart-disease.csv")

In [3]:
df.head()

## Exploratory Data Analysis
### 1. Data Overview 📋

* **Check the dimensions**: How many rows and columns are there? 📏
* **Inspect data types**: Are the columns numbers, text (strings), or dates? 🔢
* **Preview the data**: Look at the first and last few rows to get a feel for the values. 👀

### 2. Data Cleaning 🧹

* **Identify missing values**: Which columns have nulls, and why might they be missing? ❓
* **Find duplicates**: Are there any repeated rows that shouldn't be there? 👯
* **Detect outliers**: Are there extreme values that look like errors (like a human height of 10 feet)? 📈

### 3. Univariate Analysis (One variable at a time) 🧬

* **Summary statistics**: Check the mean, median, and standard deviation for numerical columns. 📊
* **Distribution**: Use histograms to see if the data is "normal" or skewed to one side. 🏔️
* **Value counts**: For categorical data (like "City" or "Color"), see which categories are the most frequent. 🏷️

### 4. Bivariate & Multivariate Analysis (Relationships) 🤝

* **Correlation**: How do numerical variables relate to each other? (e.g., Does height correlate with weight?) 🔗
* **Categorical vs. Numerical**: How does a number change across different groups? (e.g., Average salary by department). 🏦
* **Scatter plots**: Visualize the relationship between two numbers to look for patterns. 📍

### 5. Insight Synthesis 🧠

* **Feature Engineering**: Are there new columns you could create from existing ones? 🛠️
* **Hypothesis testing**: Do your initial findings support your assumptions about the data? 💡


In [4]:
fig,ax = plt.subplots(figsize=(6,5))
ax = sns.countplot(data=df, x="target",hue="target");
ax.set(title="Count of healthy and diseased people");

In [5]:
plt.figure(figsize=(9, 6))
sns.countplot(data=df, x="cp", hue="target")
plt.title("Count of Heart Disease by Chest Pain Type")

In [6]:
corr_mat = df.corr()
fig,ax = plt.subplots(figsize=(12.5,8))
sns.heatmap(data=corr_mat,annot=True,fmt=".2f",cmap="coolwarm",ax=ax);

# Column 2, 7 are positively co-related ( cp, thalach )
# Column 8, 9 are negatively co-related ( exang & oldpeak )

In [7]:
over_50 = df.filter(
    pl.col("age") > 50
)

# Initialize the figure
fig,((ax0,ax1)) = plt.subplots(figsize=(11,12),nrows=2,ncols=1,sharex=True)
fig.suptitle("Heart Disease Analysis")

# Plotting Age vs Heart-Rate
sns.scatterplot(data=over_50, x="age", y="thalach", hue="target",ax=ax0,palette={0: "blue", 1: "#32E662"})
ax0.set(title="Age vs Max Heart Rate",xlim=[50,80],ylim=[60,200])
ax0.axhline(y=over_50["thalach"].mean(), linestyle="--", color="grey")

# Plotting Age vs Cholestrol
sns.scatterplot(data=over_50,x="age",y="chol",hue="target",ax=ax1,palette={0: "blue", 1: "#32E662"})
ax1.set(title="Age vs Cholestrol")
ax1.axhline(y=over_50["chol"].mean(),linestyle="--",color="grey")


In [12]:
crosstab = (
    df.group_by(["target", "sex"])
    .agg(pl.len().alias("count"))
    .pivot(
        on="sex",
        index="target",
        values="count"
    )
    .sort("target")
)
crosstab

## Feature Engineering
* Encoding
* Scaling

In [ ]:
#

X = df.drop("target")
y = df.select(pl.col("target"))
X_train, X_test, y_test, y_train = train_test_split(X,y,test_size=0.2)
